# U-Net Droplet / Nucleus Training Notebook — v7.0

### Changes from v6.1.7

| # | Change | Why |
|---|--------|-----|
| 1 | **`droplet_focus_erosion_px` → `droplet_seed_erosion_px`** | Renamed to reflect true purpose: erodes binary droplet mask to shrink seeds pre-watershed, preventing touching droplets from merging. Used exclusively in focus-scoring functions. |
| 2 | **NPC erosion zone removed from `detect_npc_rings_per_droplet`** | NPC signal is detected across the full droplet interior. The ring zone erosion was a biological misunderstanding — NPC puncta/rings can appear anywhere inside the droplet at the nuclear envelope boundary. |
| 3 | **NPC erosion zone removed from `segment_nucleus_per_droplet`** | Per-droplet Otsu operates on the full droplet interior, not an eroded subset. |
| 4 | **Confirmed diagnostic parameters applied to config** | `droplet_local_block_size=301`, `local_offset=-0.05`, `min_droplet_area_um2=200`, `max_droplet_area_um2=1500`, `droplet_circularity_min=0.70`, `eccentricity_max=0.65`, `npc_ring_threshold_k=2.0`, `nucleus_threshold_k` removed (Otsu is self-tuning). |
| 5 | **`make_3class_label_full` removed** | Dead code — all label generation uses 4-class scheme. |
| 6 | **`segment_nucleus_from_import` retained but not called by default** | Kept for ablation; `use_per_droplet_nucleus_seg=True` remains default. |

Everything else is unchanged from v6.1.7a.


In [ ]:
# ============================================================
# 1. Core imports
# ============================================================

from dataclasses import dataclass, field
from pathlib import Path
import os, math, time, gc, shutil, inspect

import numpy as np
import pandas as pd

import tifffile as tiff
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from skimage import filters, morphology, measure
from skimage.filters import threshold_local
from skimage.measure import regionprops

import tensorflow as tf
from tensorflow.keras import layers, models

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

@dataclass
class TrainingConfig:
    # ----------------------------
    # Project paths
    # ----------------------------
    data_root: Path = Path("/home/tdeibert/Data/Machine_Learning_Dev/")
    model_root: Path = None
    image_root: Path = None
    out_root:   Path = None

    image_filename: str = "control_extract_1.1.tif"

    # ----------------------------
    # Image metadata  (T, Z, C, Y, X)
    # ----------------------------
    pixel_size_um: float = 0.108
    n_channels:    int   = 3

    membrane_channel_idx: int = 0
    nucleus_channel_idx:  int = 1
    npc_channel_idx:      int = 2

    # ----------------------------
    # Focus filtering
    # ----------------------------
    use_focus_filter:      bool  = False
    min_focus_score:       float = 0.0005
    exclude_edge_z_planes: bool  = True
    edge_z_exclusion:      int   = 2
    # Excludes coverslip artefacts near the glass at low Z planes.
    # Set focus_min_z to the first Z plane free of coverslip artefact.
    focus_min_z: int    = 6
    focus_max_z: object = None  # None = no upper bound beyond edge exclusion

    # ----------------------------
    # v7: Pre-watershed droplet seed erosion
    # Erodes the binary droplet mask before watershed to create gaps
    # between touching droplets, preventing them from being segmented
    # as a single merged object.
    # Used exclusively in focus-scoring functions (Laplacian / contrast /
    # NPC-ring scoring) to define the eroded interior region.
    # NOT used for NPC detection or nucleus segmentation.
    # ----------------------------
    droplet_seed_erosion_px: int = 10

    # ----------------------------
    # v6.6: Per-droplet focus selection (three-way adaptive)
    # ----------------------------
    use_per_droplet_focus: bool = True

    # Channel selection: nucleus Laplacian peak must exceed this floor
    # across any Z-plane to use nucleus-channel scoring.
    nucleus_score_floor:         float = 0.008
    # Above floor but below this → contrast-based scoring (late/uniform nuclei).
    nucleus_laplacian_threshold: float = 0.012

    # Per-mode minimum focus scores
    min_droplet_focus_score:    float = 0.010   # nucleus Laplacian mode
    nucleus_contrast_min_score: float = 1.5     # contrast ratio mode
    npc_ring_min_focus_score:   float = 0.001   # NPC-ring Laplacian mode

    # Jittered patches per in-focus droplet (base; scaled by timepoint weight)
    patches_per_droplet: int = 3

    # ----------------------------
    # v6.9: Nucleus presence gate
    # ----------------------------
    min_nucleus_signal_at_best_z: float = 0.0
    nucleus_signal_gate_channels: tuple = ('nucleus_laplacian', 'nucleus_contrast')

    # ----------------------------
    # v6.8: Coverslip hard-negative sampling
    # ----------------------------
    coverslip_negative_patches_per_t: int   = 20
    coverslip_z_max:                  int   = 6
    coverslip_neg_max_t:              int   = 5
    coverslip_min_brightness:         float = 0.15

    # Timepoint patch weights. None = uniform 1.0 for all timepoints.
    timepoint_patch_weights: object = None

    # ----------------------------
    # Droplet segmentation
    # Calibrated on control_extract_1.1.tif via Segmentation_Diagnostic_v1
    # ----------------------------
    use_data_driven_tail:    bool  = True
    histogram_bins:          int   = 512
    histogram_smooth_window: int   = 9
    tail_start_quantile:     float = 0.80
    min_clip_quantile:       float = 0.95
    max_clip_quantile:       float = 0.995
    puncta_clip_quantile:    float = 0.985

    droplet_blur_sigma:        float = 8.0
    droplet_local_block_size:  int   = 301    # confirmed diagnostic value
    droplet_local_offset:      float = -0.05  # confirmed diagnostic value
    min_droplet_area_um2:      float = 200.0  # confirmed diagnostic value
    max_droplet_area_um2:      float = 1500.0 # confirmed diagnostic value
    droplet_hole_area_um2:     float = 10.0   # confirmed diagnostic value
    droplet_closing_radius_um: float = 3.0    # confirmed diagnostic value
    droplet_circularity_min:   float = 0.70   # confirmed diagnostic value
    droplet_eccentricity_max:  float = 0.65   # confirmed diagnostic value

    # ----------------------------
    # Nucleus segmentation
    # ----------------------------
    min_nucleus_area_um2:      float = 50.0
    max_nucleus_area_um2:      float = 3000.0
    nucleus_blur_sigma:        float = 1.5
    nucleus_threshold_method:  str   = "otsu"
    nucleus_closing_radius_um: float = 1.5
    nucleus_hole_area_um2:     float = 200.0
    nucleus_circularity_min:   float = 0.4
    nucleus_max_droplet_frac:  float = 0.50

    # When True, Otsu threshold is computed independently per droplet interior.
    use_per_droplet_nucleus_seg: bool = True

    # ----------------------------
    # NPC detection
    # Threshold = mean + k*std across full droplet interior (no erosion zone).
    # ----------------------------
    npc_ring_threshold_k: float = 2.0  # confirmed diagnostic value

    # ----------------------------
    # Patch extraction
    # ----------------------------
    patch_size:                   int   = 512
    patch_jitter_px:              int   = 128
    patches_per_plane:            int   = 24
    nucleus_patch_fraction:       float = 0.70
    droplet_patch_fraction:       float = 0.20
    hard_negative_patch_fraction: float = 0.10
    min_label_fraction:           float = 0.005

    # ----------------------------
    # Model / training
    # ----------------------------
    model_name:          str   = "unet_droplet_nucleus_v7.0"
    num_classes:         int   = 4
    batch_size:          int   = 2
    epochs:              int   = 50
    learning_rate:       float = 1e-4
    validation_fraction: float = 0.20
    seed:                int   = 42

    # ----------------------------
    # Augmentation
    # ----------------------------
    use_augmentation: bool = True

    # ----------------------------
    # Loss class weights: [bg, droplet, npc_ring, nucleus_interior]
    # NPC ring upweighted — thin annular structure, critical for early detection
    # Nucleus interior upweighted — primary measurement target
    # ----------------------------
    loss_class_weights: tuple = (0.3, 0.5, 5.0, 4.0)

    # ----------------------------
    # Parallelism
    # ----------------------------
    # Number of parallel worker processes for patch generation.
    # -1 = use all available cores. Set to a positive int to cap.
    n_jobs: int = -1

    # Validation held-out timepoints. None -> hold out last timepoint.
    val_timepoints: object = None

    def __post_init__(self):
        if self.model_root is None:
            self.model_root = self.data_root / "Models"
        if self.image_root is None:
            self.image_root = self.data_root / "Images"
        if self.out_root is None:
            self.out_root = self.data_root / "Outputs"

    @property
    def image_file(self):
        return self.image_root / self.image_filename

    @property
    def training_root(self):
        return self.out_root / f"training_patches_{self.model_name}"

    @property
    def image_patch_dir(self):
        return self.training_root / "images"

    @property
    def label_patch_dir(self):
        return self.training_root / "labels"

    @property
    def qc_dir(self):
        return self.out_root / f"qc_{self.model_name}"

    @property
    def best_model_path(self):
        return self.model_root / f"{self.model_name}_best.keras"

    @property
    def final_model_path(self):
        return self.model_root / f"{self.model_name}_final.keras"

    def patches_for_timepoint(self, t):
        """Return patches_per_droplet scaled by timepoint weight for t."""
        if self.timepoint_patch_weights is None:
            return self.patches_per_droplet
        weights = list(self.timepoint_patch_weights)
        w = weights[t] if t < len(weights) else 1.0
        return max(1, round(self.patches_per_droplet * w))


cfg = TrainingConfig()

# ── Timepoint weights for 10 timepoints ──────────────────────────────────
# Early (t0-t2): low weight — empty/undeveloped droplets
# Mid   (t3-t5): medium weight — NPC assembly, early import
# Late  (t6-t9): high weight — fully assembled nuclei, main target
cfg.timepoint_patch_weights = (0.25, 0.25, 0.5, 0.5, 1.0, 2.0, 3.0, 4.0, 4.0, 5.0)

# Mixed validation set: early (t=2), mid (t=5), late (t=8)
cfg.val_timepoints = [2, 5, 8]

for p in [
    cfg.data_root, cfg.model_root, cfg.image_root, cfg.out_root,
    cfg.training_root, cfg.image_patch_dir, cfg.label_patch_dir, cfg.qc_dir,
]:
    p.mkdir(parents=True, exist_ok=True)

DATA_ROOT     = cfg.data_root
MODEL_ROOT    = cfg.model_root
IMAGE_ROOT    = cfg.image_root
OUT_ROOT      = cfg.out_root
IMAGE_FILE    = cfg.image_file
PIXEL_SIZE_UM = cfg.pixel_size_um
MEM_CH        = cfg.membrane_channel_idx
NUC_CH        = cfg.nucleus_channel_idx
NPC_CH        = cfg.npc_channel_idx
PATCH_SIZE    = cfg.patch_size
NUM_CHANNELS  = cfg.n_channels
NUM_CLASSES   = cfg.num_classes
BATCH_SIZE    = cfg.batch_size

print("DATA_ROOT              :", DATA_ROOT)
print("IMAGE_FILE             :", IMAGE_FILE)
print("MODEL_ROOT             :", MODEL_ROOT)
print("PATCH_SIZE             :", PATCH_SIZE)
print("Patch FOV              :", PATCH_SIZE * PIXEL_SIZE_UM, "um")
print("BEST_MODEL_PATH        :", cfg.best_model_path)
print("nucleus_score_floor    :", cfg.nucleus_score_floor)
print("laplacian_threshold    :", cfg.nucleus_laplacian_threshold)
print("min_focus_score        :", cfg.min_droplet_focus_score)
print("contrast_min_score     :", cfg.nucleus_contrast_min_score)
print("npc_ring_min_score     :", cfg.npc_ring_min_focus_score)
print("timepoint_weights      :", cfg.timepoint_patch_weights)
print("focus_min_z            :", cfg.focus_min_z)
print("focus_max_z            :", cfg.focus_max_z)
print("coverslip_z_max        :", cfg.coverslip_z_max)
print("coverslip_neg_per_t    :", cfg.coverslip_negative_patches_per_t)
print("coverslip_neg_max_t    :", cfg.coverslip_neg_max_t)
print("min_nuc_signal_bz      :", cfg.min_nucleus_signal_at_best_z)
print("nuc_gate_channels      :", cfg.nucleus_signal_gate_channels)
print("use_augmentation       :", cfg.use_augmentation)
print("droplet_seed_erosion_px:", cfg.droplet_seed_erosion_px,
      "px /", round(cfg.droplet_seed_erosion_px * cfg.pixel_size_um, 2), "um")
print("npc_ring_threshold_k   :", cfg.npc_ring_threshold_k)
print("n_jobs                 :", cfg.n_jobs)
n_t_check = len(cfg.timepoint_patch_weights) if cfg.timepoint_patch_weights else "uniform"
print(f"patches_per_droplet    : {cfg.patches_per_droplet} base  "
      f"(weighted: {[cfg.patches_for_timepoint(t) for t in range(10)]}  for t0-t9)")

In [ ]:
# ============================================================
# 3. Memory-mapped TIFF loading
# ============================================================

def load_memmap_tiff(path):
    """
    Open a memory-mapped TIFF. Returns a numpy memmap — the file is never
    fully loaded into RAM. Each worker process that calls this independently
    gets its own OS-level mapping to the same file with zero data copying.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Could not find TIFF file: {path}")
    arr = tiff.memmap(str(path))
    print("Loaded memory-mapped TIFF")
    print("  Path :", path)
    print("  Shape:", arr.shape)
    print("  Dtype:", arr.dtype)
    return arr

# Keep IMAGE_FILE as a Path so workers can re-open their own memmap
img_fov = load_memmap_tiff(IMAGE_FILE)
img = img_fov

if img_fov.ndim != 5:
    raise ValueError(f"Expected shape (T, Z, C, Y, X), got {img_fov.shape}")

print("Image shape:", img_fov.shape)
print("Image dtype:", img_fov.dtype)

In [ ]:
# ============================================================
# 4. Simple viewer: one timepoint, one Z plane, all channels
# ============================================================

def show_plane_all_channels(img, t=0, z=0, channel_names=None, percentile=(1, 99.8)):
    plane = img[t, z]
    C = plane.shape[0]
    if channel_names is None:
        channel_names = [f"Channel {i}" for i in range(C)]
    fig, axes = plt.subplots(1, C, figsize=(5 * C, 5))
    if C == 1:
        axes = [axes]
    for c in range(C):
        arr = np.asarray(plane[c])
        lo, hi = np.percentile(arr, percentile)
        axes[c].imshow(arr, cmap="gray", vmin=lo, vmax=hi)
        axes[c].set_title(f"{channel_names[c]} | t={t}, z={z}")
        axes[c].axis("off")
    plt.tight_layout()
    plt.show()

show_plane_all_channels(
    img_fov, t=0,
    z=min(10, img_fov.shape[1] - 1),
    channel_names=["Membrane / context", "Nucleus (NLS)", "NPC"],
)

In [ ]:
# ============================================================
# 5. Segmentation utilities: units, normalization, focus
# ============================================================

def area_um2_to_px(area_um2, pixel_size_um=PIXEL_SIZE_UM):
    if area_um2 is None:
        return None
    return int(np.ceil(area_um2 / (pixel_size_um ** 2)))

def radius_um_to_px(radius_um, pixel_size_um=PIXEL_SIZE_UM):
    if radius_um is None:
        return 0
    return max(1, int(np.ceil(radius_um / pixel_size_um)))

def normalize_01(img2d):
    arr = img2d.astype('float32', copy=False)
    lo, hi = np.nanmin(arr), np.nanmax(arr)
    if hi <= lo:
        return np.zeros_like(arr, dtype='float32')
    return (arr - lo) / (hi - lo)

def normalize_channel(ch2d):
    return normalize_01(ch2d).astype('float32')

def focus_score_variance_laplacian(img2d):
    """Legacy whole-plane focus score. Not used by per-droplet pipeline."""
    img_f = normalize_01(img2d)
    lap = ndi.laplace(img_f)
    return float(np.var(lap))

def z_plane_allowed(z, n_z, exclude_edge_z=cfg.edge_z_exclusion):
    if exclude_edge_z is None or exclude_edge_z <= 0:
        return True
    return exclude_edge_z <= z < (n_z - exclude_edge_z)

def z_in_focus_range(z, n_z, cfg=cfg):
    """
    Combined Z validity check for focus scoring.
    Applies edge exclusion, focus_min_z floor, and focus_max_z ceiling.
    """
    if cfg.exclude_edge_z_planes and not z_plane_allowed(z, n_z, cfg.edge_z_exclusion):
        return False
    if cfg.focus_min_z is not None and z < cfg.focus_min_z:
        return False
    if cfg.focus_max_z is not None and z > cfg.focus_max_z:
        return False
    return True

def plane_passes_focus(nuc2d, min_focus_score=cfg.min_focus_score):
    """Legacy whole-plane focus gate. Superseded by per-droplet scoring."""
    score = focus_score_variance_laplacian(nuc2d)
    return score >= min_focus_score, score

def filter_objects_by_area(mask, pixel_size_um, min_area_um2=None, max_area_um2=None):
    labeled = measure.label(mask)
    clean = np.zeros_like(mask, dtype=bool)
    min_px = 0 if min_area_um2 is None else area_um2_to_px(min_area_um2, pixel_size_um)
    max_px = np.inf if max_area_um2 is None else area_um2_to_px(max_area_um2, pixel_size_um)
    for region in measure.regionprops(labeled):
        if min_px <= region.area <= max_px:
            clean[labeled == region.label] = True
    return clean


# ── Per-droplet focus utilities (three-way adaptive) ─────────────────────

def per_droplet_focus_scores_nucleus_laplacian(nuc2d, droplet_mask,
                                               erosion_radius_px=None):
    """
    Laplacian-variance focus score on the nucleus channel interior.
    erosion_radius_px shrinks the droplet interior for scoring, providing
    a stable interior region that excludes the droplet boundary.
    Returns dict {label_id: score}.
    """
    if erosion_radius_px is None:
        erosion_radius_px = cfg.droplet_seed_erosion_px
    labeled = measure.label(droplet_mask)
    nuc_f = normalize_01(nuc2d)
    laplacian_sq = ndi.laplace(nuc_f) ** 2
    dist = ndi.distance_transform_edt(droplet_mask.astype(bool))
    eroded_mask = dist >= erosion_radius_px
    eroded_labels = labeled * eroded_mask.astype(np.int32)
    ids = np.unique(eroded_labels)[1:]
    if len(ids) == 0:
        return {}
    scores = ndi.mean(laplacian_sq, labels=eroded_labels, index=ids)
    return dict(zip(ids.tolist(), scores.tolist()))


def per_droplet_focus_scores_nucleus_contrast(nuc2d, droplet_mask,
                                              erosion_radius_px=None):
    """
    Contrast-ratio focus score: mean(interior) / mean(ring).
    erosion_radius_px defines the interior/ring boundary for scoring only.
    Returns dict {label_id: score}.
    """
    if erosion_radius_px is None:
        erosion_radius_px = cfg.droplet_seed_erosion_px
    labeled = measure.label(droplet_mask)
    nuc_f = normalize_01(nuc2d)
    dist = ndi.distance_transform_edt(droplet_mask.astype(bool))
    interior_mask = dist >= erosion_radius_px
    ring_mask     = droplet_mask.astype(bool) & ~interior_mask
    ids = np.unique(labeled)[1:]
    scores = {}
    for label_id in ids:
        droplet_interior = interior_mask & (labeled == label_id)
        droplet_ring     = ring_mask     & (labeled == label_id)
        if not droplet_interior.any() or not droplet_ring.any():
            continue
        mean_interior = float(np.mean(nuc_f[droplet_interior]))
        mean_ring     = float(np.mean(nuc_f[droplet_ring]))
        scores[int(label_id)] = mean_interior / (mean_ring + 1e-6)
    return scores


def per_droplet_focus_scores_npc_ring(npc2d, droplet_mask,
                                      erosion_radius_px=None):
    """
    Laplacian-variance focus score on the NPC boundary ring zone.
    erosion_radius_px defines the ring zone width for scoring only.
    Returns dict {label_id: score}.
    """
    if erosion_radius_px is None:
        erosion_radius_px = cfg.droplet_seed_erosion_px
    labeled = measure.label(droplet_mask)
    npc_f = normalize_01(npc2d)
    laplacian_sq = ndi.laplace(npc_f) ** 2
    dist = ndi.distance_transform_edt(droplet_mask.astype(bool))
    interior_mask = dist >= erosion_radius_px
    ring_mask     = droplet_mask.astype(bool) & ~interior_mask
    ring_labels   = labeled * ring_mask.astype(np.int32)
    ids = np.unique(ring_labels)[1:]
    if len(ids) == 0:
        return {}
    scores = ndi.mean(laplacian_sq, labels=ring_labels, index=ids)
    return dict(zip(ids.tolist(), scores.tolist()))


def droplet_has_nucleus_signal(nuc2d, droplet_binary,
                               erosion_radius_px, cfg=cfg):
    """
    Check whether the nucleus channel shows above-background signal
    inside the eroded droplet interior at a given Z plane.
    Returns (bool, float).
    """
    interior = ndi.binary_erosion(
        droplet_binary.astype(bool), iterations=erosion_radius_px)
    if not interior.any():
        return False, 0.0
    nuc_f = normalize_01(nuc2d)
    mean_intensity = float(np.mean(nuc_f[interior]))
    return mean_intensity >= cfg.min_nucleus_signal_at_best_z, mean_intensity


def select_focus_channel(img_fov, t, droplet_mask_ref, cfg=cfg):
    """
    Three-way adaptive focus channel selection for timepoint t.

    peak_lap < nucleus_score_floor
        -> 'npc_ring'           (very early: NLS absent)
    nucleus_score_floor <= peak_lap < nucleus_laplacian_threshold
        -> 'nucleus_contrast'   (late/uniform: bright disc, low texture)
    peak_lap >= nucleus_laplacian_threshold
        -> 'nucleus_laplacian'  (mid: NLS present, nuclear texture visible)

    Returns (channel_str, peak_lap_score).
    """
    n_z = img_fov.shape[1]
    peak_scores = []
    for z in range(n_z):
        if not z_in_focus_range(z, n_z, cfg=cfg):
            continue
        nuc2d = np.asarray(img_fov[t, z, cfg.nucleus_channel_idx])
        z_scores = per_droplet_focus_scores_nucleus_laplacian(
            nuc2d, droplet_mask_ref,
            erosion_radius_px=cfg.droplet_seed_erosion_px)
        if z_scores:
            peak_scores.append(max(z_scores.values()))

    peak_lap = float(np.max(peak_scores)) if peak_scores else 0.0

    if peak_lap < cfg.nucleus_score_floor:
        channel = 'npc_ring'
    elif peak_lap < cfg.nucleus_laplacian_threshold:
        channel = 'nucleus_contrast'
    else:
        channel = 'nucleus_laplacian'

    return channel, peak_lap


def select_best_focus_z_per_droplet(img_fov, t, droplet_mask_ref,
                                    erosion_radius_px=None, cfg=cfg):
    """
    Score every droplet across all valid Z-planes for timepoint t.
    Returns (dict {droplet_label_id: best_z}, channel_str, nucleus_signals_dict).
    """
    if erosion_radius_px is None:
        erosion_radius_px = cfg.droplet_seed_erosion_px

    channel, peak_lap = select_focus_channel(
        img_fov, t, droplet_mask_ref, cfg=cfg)

    min_score = {
        'npc_ring':          cfg.npc_ring_min_focus_score,
        'nucleus_contrast':  cfg.nucleus_contrast_min_score,
        'nucleus_laplacian': cfg.min_droplet_focus_score,
    }[channel]

    labeled_ref = measure.label(droplet_mask_ref)
    droplet_ids = set(np.unique(labeled_ref)[1:].tolist())
    if not droplet_ids:
        return {}, channel, {}

    n_z = img_fov.shape[1]
    score_matrix = {did: {} for did in droplet_ids}

    # Pre-compute ring/interior geometry once from reference mask
    dist_ref      = ndi.distance_transform_edt(droplet_mask_ref.astype(bool))
    interior_ref  = dist_ref >= erosion_radius_px
    ring_mask_ref = droplet_mask_ref.astype(bool) & ~interior_ref
    eroded_labels_ref = labeled_ref * interior_ref.astype(np.int32)
    ring_labels_ref   = labeled_ref * ring_mask_ref.astype(np.int32)
    ids_interior = np.unique(eroded_labels_ref)[1:]
    ids_ring     = np.unique(ring_labels_ref)[1:]

    for z in range(n_z):
        if not z_in_focus_range(z, n_z, cfg=cfg):
            continue
        if channel == 'npc_ring':
            ch2d  = np.asarray(img_fov[t, z, cfg.npc_channel_idx])
            npc_f = normalize_01(ch2d)
            lap_sq = ndi.laplace(npc_f) ** 2
            if len(ids_ring) == 0:
                z_scores = {}
            else:
                sc = ndi.mean(lap_sq, labels=ring_labels_ref, index=ids_ring)
                z_scores = dict(zip(ids_ring.tolist(), sc.tolist()))
        elif channel == 'nucleus_contrast':
            ch2d  = np.asarray(img_fov[t, z, cfg.nucleus_channel_idx])
            nuc_f = normalize_01(ch2d)
            if len(ids_interior) == 0 or len(ids_ring) == 0:
                z_scores = {}
            else:
                m_int  = ndi.mean(nuc_f, labels=eroded_labels_ref, index=ids_interior)
                m_ring = ndi.mean(nuc_f, labels=ring_labels_ref,   index=ids_ring)
                z_scores = {
                    int(did): float(mi) / (float(mr) + 1e-6)
                    for did, mi, mr in zip(ids_interior, m_int, m_ring)
                }
        else:  # nucleus_laplacian
            ch2d  = np.asarray(img_fov[t, z, cfg.nucleus_channel_idx])
            nuc_f = normalize_01(ch2d)
            lap_sq = ndi.laplace(nuc_f) ** 2
            if len(ids_interior) == 0:
                z_scores = {}
            else:
                sc = ndi.mean(lap_sq, labels=eroded_labels_ref, index=ids_interior)
                z_scores = dict(zip(ids_interior.tolist(), sc.tolist()))
        for did, score in z_scores.items():
            if did in score_matrix:
                score_matrix[did][z] = score

    labeled_ref_full = measure.label(droplet_mask_ref)

    best_z = {}
    nucleus_signals = {}
    for did, z_scores in score_matrix.items():
        if not z_scores:
            continue
        bz = max(z_scores, key=z_scores.get)
        if z_scores[bz] < min_score:
            continue
        nuc_at_bz      = np.asarray(img_fov[t, bz, cfg.nucleus_channel_idx])
        droplet_binary = labeled_ref_full == did
        _, sig = droplet_has_nucleus_signal(
            nuc_at_bz, droplet_binary, erosion_radius_px, cfg=cfg)
        nucleus_signals[int(did)] = sig
        if (cfg.min_nucleus_signal_at_best_z > 0
                and channel in cfg.nucleus_signal_gate_channels
                and not sig >= cfg.min_nucleus_signal_at_best_z):
            continue
        best_z[int(did)] = int(bz)
    return best_z, channel, nucleus_signals


def get_reference_z(img_fov, t, cfg=cfg, droplet_kwargs=None):
    """
    Fast reference Z selection: uses middle of valid Z range (one segment
    call) rather than scanning all valid planes.
    Returns (ref_z, n_droplets, droplet_mask).
    """
    dk = droplet_kwargs or {}
    n_z = img_fov.shape[1]
    valid_zs = [z for z in range(n_z) if z_in_focus_range(z, n_z, cfg=cfg)]
    if not valid_zs:
        return n_z // 2, 0, None
    ref_z = valid_zs[len(valid_zs) // 2]
    npc2d = np.asarray(img_fov[t, ref_z, cfg.npc_channel_idx])
    mask  = segment_droplet_from_npc(
        npc2d, pixel_size_um=cfg.pixel_size_um, **dk)
    n = len(np.unique(measure.label(mask))) - 1
    return ref_z, n, mask


print("50 um2 nucleus minimum =", area_um2_to_px(50, PIXEL_SIZE_UM), "px")
print("512 px patch FOV =", PATCH_SIZE * PIXEL_SIZE_UM, "um")
print("Seed erosion radius =", cfg.droplet_seed_erosion_px, "px /",
      round(cfg.droplet_seed_erosion_px * cfg.pixel_size_um, 2), "um")

In [ ]:
# ============================================================
# 6. Data-driven NPC puncta clipping
# ============================================================

def estimate_npc_histogram_tail_cutoff(
    npc2d,
    bins=cfg.histogram_bins,
    smooth_window=cfg.histogram_smooth_window,
    tail_start_quantile=cfg.tail_start_quantile,
    min_clip_quantile=cfg.min_clip_quantile,
    max_clip_quantile=cfg.max_clip_quantile,
):
    npc_f = normalize_01(npc2d)
    vals  = npc_f[np.isfinite(npc_f)].ravel()
    vals  = vals[vals > 0]

    if vals.size == 0:
        return 1.0, 1.0, {"reason": "empty_nonzero_values",
                           "cutoff_value": 1.0, "cutoff_quantile": 1.0}

    hist, edges = np.histogram(vals, bins=bins, range=(0, 1))
    centers     = (edges[:-1] + edges[1:]) / 2

    smooth_window = int(max(3, smooth_window))
    if smooth_window % 2 == 0:
        smooth_window += 1
    kernel     = np.ones(smooth_window, dtype='float32') / smooth_window
    hist_smooth = np.convolve(hist.astype('float32'), kernel, mode='same')
    log_hist    = np.log1p(hist_smooth)
    slope       = np.gradient(log_hist, centers)

    tail_start = np.quantile(vals, tail_start_quantile)
    tail_mask  = centers >= tail_start

    if tail_mask.sum() < smooth_window:
        cutoff_quantile = max_clip_quantile
        cutoff_value    = float(np.quantile(vals, cutoff_quantile))
        reason          = "fallback_insufficient_tail_bins"
    else:
        tail_centers    = centers[tail_mask]
        tail_slope      = slope[tail_mask]
        slope_smooth    = np.convolve(tail_slope, kernel, mode='same')
        tail_idx        = int(np.argmin(slope_smooth))
        raw_cutoff_val  = float(tail_centers[tail_idx])
        raw_cutoff_q    = float(np.mean(vals <= raw_cutoff_val))
        cutoff_quantile = float(np.clip(raw_cutoff_q, min_clip_quantile, max_clip_quantile))
        cutoff_value    = float(np.quantile(vals, cutoff_quantile))
        reason          = "sliding_window_histogram_tail"

    debug = {
        "hist": hist, "hist_smooth": hist_smooth, "centers": centers, "slope": slope,
        "tail_start": float(tail_start), "cutoff_value": float(cutoff_value),
        "cutoff_quantile": float(cutoff_quantile), "reason": reason,
    }
    return cutoff_value, cutoff_quantile, debug


def clip_npc_puncta_data_driven(
    npc2d,
    bins=cfg.histogram_bins,
    smooth_window=cfg.histogram_smooth_window,
    tail_start_quantile=cfg.tail_start_quantile,
    min_clip_quantile=cfg.min_clip_quantile,
    max_clip_quantile=cfg.max_clip_quantile,
):
    npc_f = normalize_01(npc2d)
    cutoff_value, cutoff_quantile, debug = estimate_npc_histogram_tail_cutoff(
        npc_f, bins=bins, smooth_window=smooth_window,
        tail_start_quantile=tail_start_quantile,
        min_clip_quantile=min_clip_quantile,
        max_clip_quantile=max_clip_quantile,
    )
    npc_clean = npc_f.copy()
    npc_clean[npc_clean > cutoff_value] = cutoff_value
    npc_clean = normalize_01(npc_clean)
    return npc_clean, cutoff_value, cutoff_quantile, debug

In [ ]:
# ============================================================
# 7. Segmentation functions
# ============================================================

def segment_droplet_from_npc(
    npc2d, pixel_size_um=PIXEL_SIZE_UM,
    use_data_driven_tail=cfg.use_data_driven_tail,
    puncta_clip_quantile=cfg.puncta_clip_quantile,
    histogram_bins=cfg.histogram_bins,
    histogram_smooth_window=cfg.histogram_smooth_window,
    tail_start_quantile=cfg.tail_start_quantile,
    min_clip_quantile=cfg.min_clip_quantile,
    max_clip_quantile=cfg.max_clip_quantile,
    blur_sigma=cfg.droplet_blur_sigma,
    local_block_size=cfg.droplet_local_block_size,
    local_offset=cfg.droplet_local_offset,
    min_droplet_area_um2=cfg.min_droplet_area_um2,
    max_droplet_area_um2=cfg.max_droplet_area_um2,
    hole_area_um2=cfg.droplet_hole_area_um2,
    closing_radius_um=cfg.droplet_closing_radius_um,
    circularity_min=cfg.droplet_circularity_min,
    eccentricity_max=cfg.droplet_eccentricity_max,
    return_debug=False, **unused_kwargs,
):
    """
    Segment oil droplets from the NPC channel.

    Pipeline:
    1. Clip NPC puncta signal (p1-p80 data-driven or fixed quantile)
    2. Gaussian blur + local adaptive threshold to detect droplet boundaries
    3. Morphological cleanup (closing, hole fill, area filter)
    4. Shape filters (circularity, eccentricity) to reject non-droplet objects

    The droplet_seed_erosion_px parameter is NOT applied here — it is used
    only in focus-scoring functions upstream.
    """
    npc_f = normalize_01(npc2d)

    if use_data_driven_tail:
        npc_clean, clip_value, clip_quantile, puncta_debug = clip_npc_puncta_data_driven(
            npc2d, bins=histogram_bins, smooth_window=histogram_smooth_window,
            tail_start_quantile=tail_start_quantile,
            min_clip_quantile=min_clip_quantile,
            max_clip_quantile=max_clip_quantile,
        )
        clip_method = "data_driven_histogram_tail"
    else:
        clip_value    = float(np.quantile(npc_f, puncta_clip_quantile))
        clip_quantile = float(puncta_clip_quantile)
        npc_clean     = npc_f.copy()
        npc_clean[npc_clean > clip_value] = clip_value
        npc_clean     = normalize_01(npc_clean)
        puncta_debug  = {"cutoff_value": clip_value, "cutoff_quantile": clip_quantile,
                         "reason": "fixed_quantile_fallback"}
        clip_method   = "fixed_quantile"

    npc_blur  = filters.gaussian(npc_clean, sigma=blur_sigma, preserve_range=True)
    npc_field = normalize_01(npc_blur)

    block_size = int(max(3, local_block_size))
    if block_size % 2 == 0:
        block_size += 1
    local_thresh  = threshold_local(npc_field, block_size=block_size, offset=local_offset)
    droplet_mask  = npc_field > local_thresh

    if closing_radius_um and closing_radius_um > 0:
        droplet_mask = morphology.binary_closing(
            droplet_mask,
            morphology.disk(radius_um_to_px(closing_radius_um, pixel_size_um)))
    if hole_area_um2 and hole_area_um2 > 0:
        droplet_mask = morphology.remove_small_holes(
            droplet_mask,
            area_threshold=area_um2_to_px(hole_area_um2, pixel_size_um))
    if min_droplet_area_um2 and min_droplet_area_um2 > 0:
        droplet_mask = morphology.remove_small_objects(
            droplet_mask,
            min_size=area_um2_to_px(min_droplet_area_um2, pixel_size_um))

    # Shape filtering: circularity and eccentricity
    labeled = measure.label(droplet_mask)
    filtered = np.zeros_like(droplet_mask, dtype=bool)
    max_area_px = area_um2_to_px(max_droplet_area_um2, pixel_size_um) if max_droplet_area_um2 else np.inf
    for region in measure.regionprops(labeled):
        if region.area > max_area_px:
            continue
        if region.perimeter == 0:
            continue
        circ = (4 * np.pi * region.area) / (region.perimeter ** 2)
        if circ < circularity_min:
            continue
        if region.eccentricity > eccentricity_max:
            continue
        filtered[labeled == region.label] = True

    droplet_mask = filtered

    if return_debug:
        return droplet_mask, {
            "npc_norm": npc_f, "npc_clean": npc_clean, "npc_blur": npc_blur,
            "npc_field": npc_field, "local_thresh": local_thresh,
            "clip_method": clip_method, "clip_value": float(clip_value),
            "clip_quantile": float(clip_quantile), "puncta_debug": puncta_debug,
            "threshold": None, "threshold_method": "local",
            "local_block_size": block_size, "local_offset": float(local_offset),
            "min_droplet_area_px": area_um2_to_px(min_droplet_area_um2, pixel_size_um),
            "unused_kwargs": unused_kwargs,
        }
    return droplet_mask


def segment_nucleus_from_import(
    nuc2d, pixel_size_um=PIXEL_SIZE_UM,
    min_nucleus_area_um2=cfg.min_nucleus_area_um2,
    max_nucleus_area_um2=cfg.max_nucleus_area_um2,
    blur_sigma=cfg.nucleus_blur_sigma,
    threshold_method=cfg.nucleus_threshold_method,
    closing_radius_um=cfg.nucleus_closing_radius_um,
    hole_area_um2=cfg.nucleus_hole_area_um2,
    return_debug=False, **unused_kwargs,
):
    """
    Global nucleus segmentation (whole-plane Otsu). Retained for ablation.
    Use segment_nucleus_per_droplet for training label generation.
    """
    nuc_f    = normalize_01(nuc2d)
    nuc_blur = filters.gaussian(nuc_f, sigma=blur_sigma, preserve_range=True)

    if threshold_method == "otsu":
        thr = filters.threshold_otsu(nuc_blur)
    elif threshold_method == "yen":
        thr = filters.threshold_yen(nuc_blur)
    elif threshold_method == "li":
        thr = filters.threshold_li(nuc_blur)
    else:
        raise ValueError("threshold_method must be 'otsu', 'yen', or 'li'.")

    nucleus_mask = nuc_blur > thr
    if closing_radius_um and closing_radius_um > 0:
        nucleus_mask = morphology.binary_closing(
            nucleus_mask,
            morphology.disk(radius_um_to_px(closing_radius_um, pixel_size_um)))
    nucleus_mask = ndi.binary_fill_holes(nucleus_mask)
    if hole_area_um2 and hole_area_um2 > 0:
        nucleus_mask = morphology.remove_small_holes(
            nucleus_mask,
            area_threshold=area_um2_to_px(hole_area_um2, pixel_size_um))
    nucleus_mask = filter_objects_by_area(
        nucleus_mask, pixel_size_um=pixel_size_um,
        min_area_um2=min_nucleus_area_um2,
        max_area_um2=max_nucleus_area_um2)
    nucleus_mask = nucleus_mask.astype(bool)

    if return_debug:
        return nucleus_mask, {
            "nuc_f": nuc_f, "nuc_blur": nuc_blur, "threshold": float(thr),
            "threshold_method": threshold_method,
            "min_nucleus_px": area_um2_to_px(min_nucleus_area_um2, pixel_size_um),
            "max_nucleus_px": area_um2_to_px(max_nucleus_area_um2, pixel_size_um),
            "unused_kwargs": unused_kwargs,
        }
    return nucleus_mask


def segment_nucleus_per_droplet(
    nuc2d, droplet_mask, pixel_size_um=PIXEL_SIZE_UM,
    min_nucleus_area_um2=cfg.min_nucleus_area_um2,
    max_nucleus_area_um2=cfg.max_nucleus_area_um2,
    blur_sigma=cfg.nucleus_blur_sigma,
    closing_radius_um=cfg.nucleus_closing_radius_um,
    hole_area_um2=cfg.nucleus_hole_area_um2,
    circularity_min=cfg.nucleus_circularity_min,
    max_droplet_frac=cfg.nucleus_max_droplet_frac,
):
    """
    Segment nuclei independently within each droplet interior.

    Computes an Otsu threshold from each droplet's full interior pixels.
    No erosion is applied — the full droplet interior is used, allowing
    NPC signal at the nuclear envelope boundary to be captured correctly
    as nucleus class rather than being masked out.

    Shape filters (circularity, max_droplet_frac) reject segmentation
    artefacts and over-segmented blobs that span the whole droplet.

    Returns nucleus_mask : 2-D bool array
    """
    labeled_droplets  = measure.label(droplet_mask)
    nucleus_mask_out  = np.zeros_like(droplet_mask, dtype=bool)
    nuc_f    = normalize_01(nuc2d)
    nuc_blur = filters.gaussian(nuc_f, sigma=blur_sigma, preserve_range=True)

    for region in measure.regionprops(labeled_droplets):
        droplet_binary  = labeled_droplets == region.label
        interior_vals   = nuc_blur[droplet_binary]
        val_range       = interior_vals.max() - interior_vals.min()
        if val_range < 1e-6:
            continue  # flat interior — no nucleus signal

        try:
            thr = filters.threshold_otsu(interior_vals)
        except Exception:
            continue

        local_mask = np.zeros_like(droplet_mask, dtype=bool)
        local_mask[droplet_binary] = nuc_blur[droplet_binary] > thr

        if closing_radius_um and closing_radius_um > 0:
            local_mask = morphology.binary_closing(
                local_mask,
                morphology.disk(radius_um_to_px(closing_radius_um, pixel_size_um)))
        local_mask = ndi.binary_fill_holes(local_mask)
        if hole_area_um2 and hole_area_um2 > 0:
            local_mask = morphology.remove_small_holes(
                local_mask,
                area_threshold=area_um2_to_px(hole_area_um2, pixel_size_um))
        local_mask = filter_objects_by_area(
            local_mask, pixel_size_um=pixel_size_um,
            min_area_um2=min_nucleus_area_um2,
            max_area_um2=max_nucleus_area_um2)

        # Shape filters on detected objects
        labeled_nuc = measure.label(local_mask)
        for nuc_region in measure.regionprops(labeled_nuc):
            # Reject objects that span too large a fraction of the droplet
            if nuc_region.area / region.area > max_droplet_frac:
                continue
            if nuc_region.perimeter == 0:
                continue
            circ = (4 * np.pi * nuc_region.area) / (nuc_region.perimeter ** 2)
            if circ < circularity_min:
                continue
            nucleus_mask_out[labeled_nuc == nuc_region.label] = True

    return nucleus_mask_out


def detect_npc_per_droplet(
    npc2d, droplet_mask, pixel_size_um=PIXEL_SIZE_UM,
    threshold_k=None,
):
    """
    Detect NPC signal per droplet by thresholding the NPC channel
    across the full droplet interior.

    Threshold = mean + k*std within each droplet's full interior,
    making detection robust to varying signal intensity across timepoints.
    Works for both punctate early NPC signal and complete late rings.

    No erosion zone is applied — NPC signal (puncta, partial rings,
    complete rings) can appear anywhere inside the droplet at the
    nuclear envelope boundary.

    Parameters
    ----------
    npc2d       : 2-D array, NPC channel
    droplet_mask: 2-D bool array, droplet segmentation
    threshold_k : float — threshold = mean + k*std per droplet interior

    Returns
    -------
    npc_mask : 2-D bool array
    """
    if threshold_k is None:
        threshold_k = cfg.npc_ring_threshold_k

    npc_f            = normalize_01(npc2d)
    labeled_droplets = measure.label(droplet_mask)
    npc_mask         = np.zeros_like(droplet_mask, dtype=bool)

    ids = np.unique(labeled_droplets)[1:]
    if len(ids) == 0:
        return npc_mask

    # Vectorised mean and std per droplet interior
    means = ndi.mean(npc_f,               labels=labeled_droplets, index=ids)
    stds  = ndi.standard_deviation(npc_f, labels=labeled_droplets, index=ids)

    for i, did in enumerate(ids):
        thr          = means[i] + threshold_k * stds[i]
        droplet_zone = labeled_droplets == did
        npc_mask[droplet_zone] = npc_f[droplet_zone] > thr

    return npc_mask


def make_4class_label(
    npc2d, nuc2d, droplet_mask, pixel_size_um=PIXEL_SIZE_UM,
    nucleus_kwargs=None, threshold_k=None,
    enforce_nucleus_inside_droplet=True,
):
    """
    Build a 4-class label map.

    Classes
    -------
    0 : Background (outside all droplets)
    1 : Droplet interior
    2 : NPC signal (puncta / partial ring / complete ring on nuclear envelope)
    3 : Nucleus interior (NLS-defined ROI)

    All assignments are purely signal-driven with no timepoint gating.
    Label priority (higher class overwrites lower):
      1 (droplet) -> 2 (NPC) -> 3 (nucleus)
    """
    nucleus_kwargs = nucleus_kwargs or {}
    nk_filtered = {k: v for k, v in nucleus_kwargs.items()
                   if k in ('min_nucleus_area_um2', 'max_nucleus_area_um2',
                            'blur_sigma', 'closing_radius_um', 'hole_area_um2')}

    label = np.zeros(droplet_mask.shape, dtype=np.uint8)

    # Class 1: droplet interior — base layer
    label[droplet_mask] = 1

    # Class 2: NPC signal — full interior, no erosion zone
    npc_mask = detect_npc_per_droplet(
        npc2d, droplet_mask, pixel_size_um=pixel_size_um,
        threshold_k=threshold_k)
    label[npc_mask] = 2

    # Class 3: nucleus interior — per-droplet Otsu on NLS channel
    if cfg.use_per_droplet_nucleus_seg:
        nucleus_mask = segment_nucleus_per_droplet(
            nuc2d, droplet_mask, pixel_size_um=pixel_size_um, **nk_filtered)
    else:
        nucleus_mask = segment_nucleus_from_import(
            nuc2d, pixel_size_um=pixel_size_um, **nucleus_kwargs)

    if enforce_nucleus_inside_droplet:
        nucleus_mask = nucleus_mask & droplet_mask
    label[nucleus_mask] = 3

    return label, npc_mask, nucleus_mask


def build_training_label_plane(
    npc2d, nuc2d, pixel_size_um=PIXEL_SIZE_UM,
    min_focus_score=cfg.min_focus_score,
    use_focus_filter=cfg.use_focus_filter,
    droplet_kwargs=None, nucleus_kwargs=None,
    enforce_nucleus_inside_droplet=True,
):
    """
    Build a 4-class label map for one Z-plane.
    Focus selection is applied upstream at the droplet level;
    use_focus_filter=False is the normal path from the patch builder.
    """
    droplet_kwargs = droplet_kwargs or {}
    nucleus_kwargs = nucleus_kwargs or {}

    focus_pass, focus_score = plane_passes_focus(nuc2d, min_focus_score=min_focus_score)
    if use_focus_filter and not focus_pass:
        return None, {"focus_pass": False, "focus_score": focus_score,
                      "droplet_area_px": 0, "nucleus_area_px": 0,
                      "reason": "out_of_focus"}, None

    droplet_mask, droplet_debug = segment_droplet_from_npc(
        npc2d, pixel_size_um=pixel_size_um, return_debug=True, **droplet_kwargs)

    label, npc_mask, nucleus_mask = make_4class_label(
        npc2d=npc2d, nuc2d=nuc2d,
        droplet_mask=droplet_mask,
        pixel_size_um=pixel_size_um,
        nucleus_kwargs=nucleus_kwargs,
        enforce_nucleus_inside_droplet=enforce_nucleus_inside_droplet,
    )

    metadata = {
        "focus_pass": True, "focus_score": focus_score,
        "droplet_area_px":  int(droplet_mask.sum()),
        "npc_area_px":      int(npc_mask.sum()),
        "nucleus_area_px":  int(nucleus_mask.sum()),
        "npc_clip_method":  droplet_debug.get("clip_method"),
        "npc_clip_value":   droplet_debug.get("clip_value"),
        "npc_clip_quantile":droplet_debug.get("clip_quantile"),
        "threshold_method": droplet_debug.get("threshold_method"),
        "local_block_size": droplet_debug.get("local_block_size"),
        "local_offset":     droplet_debug.get("local_offset"),
        "reason": "included",
    }
    debug = {**droplet_debug,
             "droplet_mask": droplet_mask,
             "npc_mask":     npc_mask,
             "nucleus_mask": nucleus_mask}
    return label, metadata, debug


print("Active droplet function:", inspect.signature(segment_droplet_from_npc))
print("Active nucleus function:", inspect.signature(segment_nucleus_per_droplet))
print("Active NPC function    :", inspect.signature(detect_npc_per_droplet))

In [ ]:
# ============================================================
# 5c. QC: per-droplet focus score distribution across Z
# ============================================================

def plot_droplet_focus_scores(img_fov, t=0, cfg=cfg):
    n_z   = img_fov.shape[1]
    mid_z = n_z // 2

    npc_ref         = np.asarray(img_fov[t, mid_z, cfg.npc_channel_idx])
    droplet_mask_ref = segment_droplet_from_npc(npc_ref, pixel_size_um=cfg.pixel_size_um)
    labeled_ref     = measure.label(droplet_mask_ref)
    droplet_ids     = np.unique(labeled_ref)[1:]

    if len(droplet_ids) == 0:
        print(f't={t}: no droplets detected in reference plane.')
        return

    channel, peak_lap = select_focus_channel(img_fov, t, droplet_mask_ref, cfg=cfg)
    min_score = {
        'npc_ring':          cfg.npc_ring_min_focus_score,
        'nucleus_contrast':  cfg.nucleus_contrast_min_score,
        'nucleus_laplacian': cfg.min_droplet_focus_score,
    }[channel]
    n_patches = cfg.patches_for_timepoint(t)
    print(f't={t}: channel={channel!r}  peak_lap={peak_lap:.4f}  '
          f'threshold={min_score}  patches_per_droplet={n_patches}  '
          f'n_droplets={len(droplet_ids)}  '
          f'min_nuc_signal={cfg.min_nucleus_signal_at_best_z}')

    score_matrix = np.full((len(droplet_ids), n_z), np.nan)
    id_to_row    = {int(did): i for i, did in enumerate(droplet_ids)}

    for z in range(n_z):
        if not z_in_focus_range(z, n_z, cfg=cfg):
            continue
        if channel == 'npc_ring':
            ch2d    = np.asarray(img_fov[t, z, cfg.npc_channel_idx])
            z_scores = per_droplet_focus_scores_npc_ring(
                ch2d, droplet_mask_ref,
                erosion_radius_px=cfg.droplet_seed_erosion_px)
        elif channel == 'nucleus_contrast':
            ch2d    = np.asarray(img_fov[t, z, cfg.nucleus_channel_idx])
            z_scores = per_droplet_focus_scores_nucleus_contrast(
                ch2d, droplet_mask_ref,
                erosion_radius_px=cfg.droplet_seed_erosion_px)
        else:
            ch2d    = np.asarray(img_fov[t, z, cfg.nucleus_channel_idx])
            z_scores = per_droplet_focus_scores_nucleus_laplacian(
                ch2d, droplet_mask_ref,
                erosion_radius_px=cfg.droplet_seed_erosion_px)
        for did, score in z_scores.items():
            row = id_to_row.get(int(did))
            if row is not None:
                score_matrix[row, z] = score

    best_scores = np.nanmax(score_matrix, axis=1)
    best_zs     = np.nanargmax(score_matrix, axis=1)

    labeled_ref_qc  = measure.label(droplet_mask_ref)
    nuc_signals_qc  = []
    for row, did in enumerate(droplet_ids):
        if np.isfinite(best_scores[row]) and best_scores[row] >= min_score:
            bz       = int(best_zs[row])
            nuc2d_bz = np.asarray(img_fov[t, bz, cfg.nucleus_channel_idx])
            _, sig   = droplet_has_nucleus_signal(
                nuc2d_bz, labeled_ref_qc == did,
                cfg.droplet_seed_erosion_px, cfg=cfg)
            nuc_signals_qc.append(sig)
        else:
            nuc_signals_qc.append(np.nan)
    nuc_signals_qc = np.array(nuc_signals_qc)

    fig, axes = plt.subplots(1, 4, figsize=(24, 5))
    fig.suptitle(f't={t} | channel: {channel!r} | peak_lap: {peak_lap:.4f} | '
                 f'patches/droplet: {n_patches} | '
                 f'min_nuc_signal: {cfg.min_nucleus_signal_at_best_z}', fontsize=10)

    z_range = np.arange(n_z)
    for row in range(len(droplet_ids)):
        axes[0].plot(z_range, score_matrix[row], alpha=0.4, linewidth=0.8)
    axes[0].axhline(min_score, color='red', linestyle='--',
                    label=f'threshold = {min_score}')
    axes[0].set_xlabel('Z plane')
    axes[0].set_ylabel('Focus score')
    axes[0].set_title(f'Focus score vs Z  (t={t}, {channel})')
    axes[0].legend()

    axes[1].hist(best_scores[np.isfinite(best_scores)], bins=30, edgecolor='k')
    axes[1].axvline(min_score, color='red', linestyle='--',
                    label=f'focus threshold = {min_score}')
    axes[1].set_xlabel('Best-Z focus score')
    axes[1].set_ylabel('Droplet count')
    axes[1].set_title('Distribution of best focus scores')
    axes[1].legend()

    best_z_map = np.full(droplet_mask_ref.shape, np.nan)
    for row, did in enumerate(droplet_ids):
        if np.isfinite(best_scores[row]) and best_scores[row] >= min_score:
            best_z_map[labeled_ref == did] = best_zs[row]
    im = axes[2].imshow(best_z_map, cmap='viridis')
    plt.colorbar(im, ax=axes[2], label='Best-focus Z index')
    axes[2].set_title('Best-focus Z per droplet (spatial)')
    axes[2].axis('off')

    valid_sigs = nuc_signals_qc[np.isfinite(nuc_signals_qc)]
    if len(valid_sigs) > 0:
        axes[3].hist(valid_sigs, bins=30, edgecolor='k')
        axes[3].axvline(cfg.min_nucleus_signal_at_best_z, color='red', linestyle='--',
                        label=f'gate = {cfg.min_nucleus_signal_at_best_z}')
        axes[3].set_xlabel('Mean nucleus intensity at best-Z (normalized)')
        axes[3].set_ylabel('Droplet count')
        axes[3].set_title('Nucleus signal at best-Z\n(tune min_nucleus_signal_at_best_z)')
        axes[3].legend()
        n_would_pass = int(np.sum(valid_sigs >= cfg.min_nucleus_signal_at_best_z))
        print(f'  Signal gate would accept: {n_would_pass} / {len(valid_sigs)} droplets')
    else:
        axes[3].text(0.5, 0.5, 'No valid signals', ha='center', va='center',
                     transform=axes[3].transAxes)

    plt.tight_layout()
    plt.show()

    accepted = int(np.sum(best_scores >= min_score))
    print(f'  Focus accepted: {accepted} / {len(droplet_ids)}')
    if np.any(np.isfinite(best_zs)):
        print(f'  Best-Z range: {int(np.nanmin(best_zs))} - {int(np.nanmax(best_zs))}')


n_t_total = img_fov.shape[0]
for t_qc in range(n_t_total):
    plot_droplet_focus_scores(img_fov, t=t_qc, cfg=cfg)

In [ ]:
# ============================================================
# 8. QC: build and visualize one training plane
# ============================================================

t = 9 if img_fov.shape[0] > 9 else img_fov.shape[0] - 1
z = min(15, img_fov.shape[1] - 1)  # default to t=7 z=15 diagnostic plane

npc2d = img_fov[t, z, NPC_CH, :, :]
nuc2d = img_fov[t, z, NUC_CH, :, :]

label_test, metadata_test, debug_test = build_training_label_plane(
    npc2d=npc2d, nuc2d=nuc2d, pixel_size_um=PIXEL_SIZE_UM,
    droplet_kwargs={
        "use_data_driven_tail":    cfg.use_data_driven_tail,
        "histogram_bins":          cfg.histogram_bins,
        "histogram_smooth_window": cfg.histogram_smooth_window,
        "tail_start_quantile":     cfg.tail_start_quantile,
        "min_clip_quantile":       cfg.min_clip_quantile,
        "max_clip_quantile":       cfg.max_clip_quantile,
        "blur_sigma":              cfg.droplet_blur_sigma,
        "local_block_size":        cfg.droplet_local_block_size,
        "local_offset":            cfg.droplet_local_offset,
        "min_droplet_area_um2":    cfg.min_droplet_area_um2,
        "max_droplet_area_um2":    cfg.max_droplet_area_um2,
        "hole_area_um2":           cfg.droplet_hole_area_um2,
        "closing_radius_um":       cfg.droplet_closing_radius_um,
        "circularity_min":         cfg.droplet_circularity_min,
        "eccentricity_max":        cfg.droplet_eccentricity_max,
    },
    nucleus_kwargs={
        "min_nucleus_area_um2":    cfg.min_nucleus_area_um2,
        "max_nucleus_area_um2":    cfg.max_nucleus_area_um2,
        "blur_sigma":              cfg.nucleus_blur_sigma,
        "closing_radius_um":       cfg.nucleus_closing_radius_um,
        "hole_area_um2":           cfg.nucleus_hole_area_um2,
    },
)

print(metadata_test)
if label_test is not None:
    print('Label classes/counts:', dict(zip(*np.unique(label_test, return_counts=True))))

    fig, ax = plt.subplots(2, 3, figsize=(15, 9))
    ax[0, 0].imshow(npc2d,  cmap="gray");                            ax[0, 0].set_title("Raw NPC channel")
    ax[0, 1].imshow(debug_test["npc_blur"],  cmap="gray");           ax[0, 1].set_title("NPC clipped + blur")
    ax[0, 2].imshow(debug_test["npc_field"], cmap="gray");           ax[0, 2].set_title("Flat-field corrected NPC")
    ax[1, 0].imshow(nuc2d,  cmap="gray");                            ax[1, 0].set_title("Raw NLS channel")
    ax[1, 1].imshow(debug_test["nucleus_mask"], cmap="gray");        ax[1, 1].set_title("Nucleus mask (class 3)")
    ax[1, 2].imshow(label_test, cmap="viridis", vmin=0, vmax=3);    ax[1, 2].set_title("Label: 0=bg 1=droplet 2=NPC 3=nucleus")

    fig2, ax2 = plt.subplots(1, 3, figsize=(15, 5))
    ax2[0].imshow(debug_test["droplet_mask"], cmap="gray");          ax2[0].set_title("Droplet mask (class 1)")
    ax2[1].imshow(debug_test["npc_mask"],     cmap="gray");          ax2[1].set_title("NPC mask (class 2) — full interior")
    ax2[2].imshow(label_test, cmap="viridis", vmin=0, vmax=3);      ax2[2].set_title("Final 4-class label")
    for a in list(ax.ravel()) + list(ax2.ravel()):
        a.axis("off")
    classes, counts = np.unique(label_test, return_counts=True)
    print("Class counts:", {int(c): int(n) for c, n in zip(classes, counts)})
    print("  0=bg  1=droplet  2=NPC  3=nucleus")
    plt.tight_layout()
    plt.show()
else:
    print('Plane excluded:', metadata_test)

In [ ]:
# ============================================================
# 9. Patch extraction helpers
# ============================================================

def build_input_patch(mem_patch, nuc_patch, npc_patch):
    """Returns H x W x 3 float32 in [0,1]. Channel order: nucleus, NPC, membrane."""
    return np.stack([
        normalize_channel(nuc_patch),
        normalize_channel(npc_patch),
        normalize_channel(mem_patch),
    ], axis=-1).astype('float32')

def clamp_patch_center(cy, cx, height, width, patch_size):
    half = patch_size // 2
    cy = int(np.clip(cy, half, height - half))
    cx = int(np.clip(cx, half, width  - half))
    return cy, cx

def extract_2d_patch(arr2d, cy, cx, patch_size):
    half = patch_size // 2
    return arr2d[cy - half:cy + half, cx - half:cx + half]

def extract_plane_patch(img_fov, t, z, cy, cx, patch_size=cfg.patch_size):
    mem_patch = extract_2d_patch(img_fov[t, z, MEM_CH], cy, cx, patch_size)
    nuc_patch = extract_2d_patch(img_fov[t, z, NUC_CH], cy, cx, patch_size)
    npc_patch = extract_2d_patch(img_fov[t, z, NPC_CH], cy, cx, patch_size)
    return build_input_patch(mem_patch, nuc_patch, npc_patch)

def centers_from_mask(mask, max_centers=None, rng=None):
    rng = rng or np.random.default_rng(SEED)
    labeled = measure.label(mask)
    centers = [(int(r.centroid[0]), int(r.centroid[1])) for r in measure.regionprops(labeled)]
    if max_centers is not None and len(centers) > max_centers:
        idx = rng.choice(len(centers), size=max_centers, replace=False)
        centers = [centers[i] for i in idx]
    return centers

def jitter_center(cy, cx, jitter_px, height, width, patch_size, rng):
    if jitter_px and jitter_px > 0:
        cy += int(rng.integers(-jitter_px, jitter_px + 1))
        cx += int(rng.integers(-jitter_px, jitter_px + 1))
    return clamp_patch_center(cy, cx, height, width, patch_size)

def random_centers_from_mask(mask, n, height, width, patch_size, rng):
    ys, xs = np.where(mask)
    if len(ys) == 0 or n <= 0:
        return []
    idx = rng.choice(len(ys), size=min(n, len(ys)), replace=False)
    return [clamp_patch_center(int(ys[i]), int(xs[i]), height, width, patch_size)
            for i in idx]

def get_completed_timepoints(cfg):
    """Check per-timepoint sentinel files for resume support."""
    completed = set()
    for flag in cfg.training_root.glob('t???_complete.flag'):
        try:
            t = int(flag.stem.split('_')[0][1:])
            completed.add(t)
        except (ValueError, IndexError):
            pass
    return completed

def write_timepoint_sentinel(cfg, t, n_patches):
    sentinel = cfg.training_root / f't{t:03d}_complete.flag'
    sentinel.write_text(f't={t} patches={n_patches}')

def clear_timepoint_sentinel(cfg, t):
    sentinel = cfg.training_root / f't{t:03d}_complete.flag'
    if sentinel.exists():
        sentinel.unlink()

In [ ]:
# ============================================================
# 10. Build training patches — v7.1 parallelised per-droplet builder
# ============================================================

from joblib import Parallel, delayed
import hashlib

def _droplet_kwargs(cfg):
    return {
        "use_data_driven_tail":    cfg.use_data_driven_tail,
        "histogram_bins":          cfg.histogram_bins,
        "histogram_smooth_window": cfg.histogram_smooth_window,
        "tail_start_quantile":     cfg.tail_start_quantile,
        "min_clip_quantile":       cfg.min_clip_quantile,
        "max_clip_quantile":       cfg.max_clip_quantile,
        "blur_sigma":              cfg.droplet_blur_sigma,
        "local_block_size":        cfg.droplet_local_block_size,
        "local_offset":            cfg.droplet_local_offset,
        "min_droplet_area_um2":    cfg.min_droplet_area_um2,
        "max_droplet_area_um2":    cfg.max_droplet_area_um2,
        "hole_area_um2":           cfg.droplet_hole_area_um2,
        "closing_radius_um":       cfg.droplet_closing_radius_um,
        "circularity_min":         cfg.droplet_circularity_min,
        "eccentricity_max":        cfg.droplet_eccentricity_max,
    }

def _nucleus_kwargs(cfg):
    return {
        "min_nucleus_area_um2":    cfg.min_nucleus_area_um2,
        "max_nucleus_area_um2":    cfg.max_nucleus_area_um2,
        "blur_sigma":              cfg.nucleus_blur_sigma,
        "closing_radius_um":       cfg.nucleus_closing_radius_um,
        "hole_area_um2":           cfg.nucleus_hole_area_um2,
    }


def _patch_stem(t, z, did, cy, cx):
    """
    Unique patch filename stem derived entirely from (t, z, droplet, y, x).
    No global counter needed — safe for parallel workers.
    A 4-char hash guards against the rare jitter collision where two patches
    land on the same pixel center.
    """
    raw = f"{t}_{z}_{did}_{cy}_{cx}"
    h   = hashlib.sha1(raw.encode()).hexdigest()[:4]
    return f"t{t:03d}_z{z:03d}_d{did:04d}_y{cy:04d}_x{cx:04d}_{h}"


def _process_timepoint(image_file, t, cfg, overwrite=False):
    """
    Process a single timepoint: focus scoring, label generation,
    patch extraction, coverslip negatives, sentinel writing.

    Accepts image_file as a Path so each worker opens its own memmap
    independently — no array pickling across process boundaries.

    Returns list of summary dicts for this timepoint.
    """
    import tifffile as tiff_w
    import numpy as np_w
    from pathlib import Path as Path_w
    from scipy import ndimage as ndi_w
    from skimage import filters as filters_w, morphology as morph_w, measure as measure_w
    import gc

    # Re-open memmap in this worker process
    img = tiff_w.memmap(str(image_file))

    dk  = _droplet_kwargs(cfg)
    nk  = _nucleus_kwargs(cfg)
    rng = np_w.random.default_rng(cfg.seed + t)  # deterministic per-timepoint seed

    n_t, n_z, n_c, height, width = img.shape
    summary_rows = []

    # Resume check
    sentinel = cfg.training_root / f't{t:03d}_complete.flag'
    if not overwrite and sentinel.exists():
        print(f'  t={t}: already complete, skipping.')
        return summary_rows

    # ── Step 1: reference droplet layout ─────────────────────────────────
    ref_z, n_droplets, droplet_mask_ref = get_reference_z(
        img, t, cfg=cfg, droplet_kwargs=dk)
    if droplet_mask_ref is None:
        npc_ref         = np_w.asarray(img[t, ref_z, cfg.npc_channel_idx])
        droplet_mask_ref = segment_droplet_from_npc(
            npc_ref, pixel_size_um=cfg.pixel_size_um, **dk)

    labeled_ref = measure_w.label(droplet_mask_ref)
    n_droplets  = len(np_w.unique(labeled_ref)) - 1

    if n_droplets == 0:
        print(f"  t={t}: no droplets at ref z={ref_z}, skipping.")
        write_timepoint_sentinel(cfg, t, 0)
        return summary_rows

    # ── Steps 2 & 3: best-focus Z per droplet ────────────────────────────
    best_z_per_droplet, focus_channel, nuc_signals = select_best_focus_z_per_droplet(
        img, t, droplet_mask_ref,
        erosion_radius_px=cfg.droplet_seed_erosion_px, cfg=cfg)

    n_accepted  = len(best_z_per_droplet)
    n_patches_t = cfg.patches_for_timepoint(t)
    print(f"  t={t}: ref_z={ref_z}  channel={focus_channel!r}  "
          f"{n_accepted}/{n_droplets} droplets accepted  "
          f"patches_per_droplet={n_patches_t}")

    # ── Step 4: group droplets by best Z ─────────────────────────────────
    z_to_droplets = {}
    for did, bz in best_z_per_droplet.items():
        z_to_droplets.setdefault(bz, []).append(did)

    region_centroids = {
        int(r.label): (int(r.centroid[0]), int(r.centroid[1]))
        for r in measure_w.regionprops(labeled_ref)
    }

    # ── Step 5: extract patches — one label plane per unique Z ───────────
    for z, droplet_ids in z_to_droplets.items():
        npc2d = np_w.asarray(img[t, z, cfg.npc_channel_idx])
        nuc2d = np_w.asarray(img[t, z, cfg.nucleus_channel_idx])

        label, metadata, debug = build_training_label_plane(
            npc2d=npc2d, nuc2d=nuc2d,
            pixel_size_um=cfg.pixel_size_um,
            use_focus_filter=False,
            droplet_kwargs=dk,
            nucleus_kwargs=nk,
        )
        if label is None or debug['droplet_mask'].sum() == 0:
            del label, debug
            gc.collect()
            continue

        for did in droplet_ids:
            if did not in region_centroids:
                continue
            cy_base, cx_base = region_centroids[did]
            patches_saved = 0

            for _ in range(n_patches_t):
                cy, cx = jitter_center(
                    cy_base, cx_base,
                    cfg.patch_jitter_px,
                    height, width, cfg.patch_size, rng)

                y_patch = extract_2d_patch(label, cy, cx, cfg.patch_size)
                if y_patch.shape != (cfg.patch_size, cfg.patch_size):
                    continue
                if np_w.mean(y_patch > 0) < cfg.min_label_fraction:
                    continue

                x_patch = extract_plane_patch(img, t, z, cy, cx, cfg.patch_size)
                stem    = _patch_stem(t, z, did, cy, cx)
                np_w.save(cfg.image_patch_dir / f'img_{stem}.npy', x_patch.astype('float32'))
                np_w.save(cfg.label_patch_dir / f'lab_{stem}.npy', y_patch.astype('uint8'))
                patches_saved += 1

            summary_rows.append({
                "t": t, "z": z, "droplet_id": did,
                "focus_channel":            focus_channel,
                "patch_type":               "focus_selected",
                "nucleus_signal_at_best_z": nuc_signals.get(did, float('nan')),
                "npc_area_px":              int(debug["npc_mask"].sum()),
                "nucleus_area_px":          int(debug["nucleus_mask"].sum()),
                "droplet_area_px":          int(debug["droplet_mask"].sum()),
                "patches_saved":            patches_saved,
            })

        del label, debug
        gc.collect()

    # ── Step 6: coverslip hard-negative patches ───────────────────────────
    if cfg.coverslip_negative_patches_per_t > 0 and t <= cfg.coverslip_neg_max_t:
        neg_rows, n_neg = _sample_coverslip_negatives_worker(img, t, cfg, rng, dk)
        summary_rows.extend(neg_rows)
        print(f"    t={t} coverslip negatives: {n_neg} patches")

    # ── Step 7: sentinel ─────────────────────────────────────────────────
    t_patches = sum(r.get('patches_saved', 0) for r in summary_rows)
    write_timepoint_sentinel(cfg, t, t_patches)
    print(f"  t={t}: sentinel written ({t_patches} patches total)")

    return summary_rows


def _sample_coverslip_negatives_worker(img, t, cfg, rng, dk):
    """
    Coverslip negative sampling — worker-safe version.
    Uses _patch_stem for filenames instead of a global patch_id counter.
    """
    import numpy as np_w
    from skimage import measure as measure_w
    import gc

    n_t, n_z, n_c, height, width = img.shape
    rows = []

    z_floor      = cfg.edge_z_exclusion if cfg.exclude_edge_z_planes else 0
    coverslip_zs = [z for z in range(z_floor, cfg.coverslip_z_max) if z < n_z]
    if not coverslip_zs:
        return rows, 0

    patches_saved = 0
    attempts      = 0
    max_attempts  = cfg.coverslip_negative_patches_per_t * 10

    while patches_saved < cfg.coverslip_negative_patches_per_t and attempts < max_attempts:
        attempts += 1
        z     = int(rng.choice(coverslip_zs))
        npc2d = np_w.asarray(img[t, z, cfg.npc_channel_idx])
        nuc2d = np_w.asarray(img[t, z, cfg.nucleus_channel_idx])

        droplet_mask = segment_droplet_from_npc(
            npc2d, pixel_size_um=cfg.pixel_size_um, **dk)
        if not droplet_mask.any():
            continue

        label = np_w.zeros((height, width), dtype=np_w.uint8)
        label[droplet_mask] = 1

        labeled_cs = measure_w.label(droplet_mask)
        regions    = measure_w.regionprops(labeled_cs)
        if not regions:
            continue

        region  = regions[int(rng.integers(0, len(regions)))]
        cy_base = int(region.centroid[0])
        cx_base = int(region.centroid[1])
        cy, cx  = jitter_center(
            cy_base, cx_base, cfg.patch_jitter_px,
            height, width, cfg.patch_size, rng)

        y_patch = extract_2d_patch(label, cy, cx, cfg.patch_size)
        if y_patch.shape != (cfg.patch_size, cfg.patch_size):
            continue
        if np_w.mean(y_patch > 0) < cfg.min_label_fraction:
            continue

        x_patch        = extract_plane_patch(img, t, z, cy, cx, cfg.patch_size)
        nuc_patch_norm = normalize_01(x_patch[..., 0])
        if float(np_w.mean(nuc_patch_norm)) < cfg.coverslip_min_brightness:
            continue

        stem = _patch_stem(t, z, int(region.label), cy, cx) + "_cneg"
        np_w.save(cfg.image_patch_dir / f'img_{stem}.npy', x_patch.astype('float32'))
        np_w.save(cfg.label_patch_dir / f'lab_{stem}.npy', y_patch.astype('uint8'))
        patches_saved += 1

        rows.append({
            't': t, 'z': z, 'droplet_id': int(region.label),
            'focus_channel': 'coverslip_negative',
            'patch_type':    'coverslip_negative',
            'nucleus_area_px': 0,
            'droplet_area_px': int(droplet_mask.sum()),
            'patches_saved':   1,
        })

    return rows, patches_saved


def build_training_patches_per_droplet_focus(img_fov, cfg=cfg, overwrite=False):
    """
    v7.1 parallelised patch builder.

    Each timepoint is processed by an independent worker process that opens
    its own memmap to the source TIFF. No array data is pickled or shared
    across process boundaries. Patch files are written directly to disk by
    each worker. Summary rows are collected and merged after all workers
    complete.

    Parallelism is controlled by cfg.n_jobs (-1 = all cores).
    """
    if overwrite:
        for d in [cfg.image_patch_dir, cfg.label_patch_dir]:
            if d.exists():
                shutil.rmtree(d)
            d.mkdir(parents=True, exist_ok=True)
        for flag in cfg.training_root.glob('t???_complete.flag'):
            flag.unlink()
    else:
        cfg.image_patch_dir.mkdir(parents=True, exist_ok=True)
        cfg.label_patch_dir.mkdir(parents=True, exist_ok=True)

    n_t = img_fov.shape[0]

    # Determine which timepoints still need processing
    completed_t = get_completed_timepoints(cfg) if not overwrite else set()
    pending_t   = [t for t in range(n_t) if t not in completed_t]

    if not pending_t:
        print("All timepoints already complete.")
    else:
        skipped = sorted(completed_t)
        if skipped:
            print(f"Resuming — skipping completed timepoints: {skipped}")
        print(f"Processing timepoints: {pending_t}  (n_jobs={cfg.n_jobs})")

    # ── Parallel dispatch ─────────────────────────────────────────────────
    # Pass IMAGE_FILE (Path) not img_fov (array) — each worker re-opens
    # its own memmap, avoiding pickling a large numpy array.
    all_rows = Parallel(n_jobs=cfg.n_jobs, backend='loky', verbose=10)(
        delayed(_process_timepoint)(IMAGE_FILE, t, cfg, overwrite)
        for t in pending_t
    )

    # Flatten per-timepoint row lists
    summary_rows = [row for rows in all_rows for row in rows]

    summary_df   = pd.DataFrame(summary_rows)
    summary_path = cfg.training_root / "training_patch_summary_v7.csv"
    summary_df.to_csv(summary_path, index=False)

    total_patches = int(summary_df['patches_saved'].sum()) if not summary_df.empty else 0
    print(f"\nTotal patches saved : {total_patches}")
    print(f"Summary CSV         : {summary_path}")

    if not summary_df.empty:
        print("\nPer-timepoint patch counts:")
        for tt in sorted(summary_df['t'].unique()):
            sub = summary_df[summary_df['t'] == tt]
            fs  = sub[sub['patch_type'] == 'focus_selected']['patches_saved'].sum()                   if 'patch_type' in sub.columns else sub['patches_saved'].sum()
            cn  = sub[sub['patch_type'] == 'coverslip_negative']['patches_saved'].sum()                   if 'patch_type' in sub.columns else 0
            print(f"  t={tt}: {int(fs)} focus-selected  {int(cn)} coverslip-negative")

    return summary_df


# ── Run patch generation ──────────────────────────────────────────────────
# Set overwrite=True on first run; False to resume from sentinels.
summary_df = build_training_patches_per_droplet_focus(img_fov, cfg=cfg, overwrite=False)

In [ ]:
# ============================================================
# 11. Patch QC
# ============================================================

def list_patch_files(cfg=cfg):
    img_paths = sorted(cfg.image_patch_dir.glob('*.npy'))
    lab_paths = sorted(cfg.label_patch_dir.glob('*.npy'))
    print(f"Found {len(img_paths)} image patches and {len(lab_paths)} label patches.")
    return img_paths, lab_paths


def show_random_patch(cfg=cfg, seed=SEED):
    img_paths, lab_paths = list_patch_files(cfg)
    if not img_paths:
        print("No patches found. Run build_training_patches_per_droplet_focus first.")
        return
    rng = np.random.default_rng(seed)
    i   = int(rng.integers(0, len(img_paths)))
    x   = np.load(img_paths[i])
    y   = np.load(lab_paths[i])
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(x[..., 0], cmap="gray");             ax[0].set_title("Input ch0: NLS (nucleus)")
    ax[1].imshow(x[..., 1], cmap="gray");             ax[1].set_title("Input ch1: NPC")
    ax[2].imshow(x[..., 2], cmap="gray");             ax[2].set_title("Input ch2: Membrane")
    ax[3].imshow(y, cmap="viridis", vmin=0, vmax=3);  ax[3].set_title("Label: 0=bg 1=droplet 2=NPC 3=nucleus")
    for a in ax:
        a.axis("off")
    plt.tight_layout()
    plt.show()
    print(img_paths[i].name)
    counts = dict(zip(*np.unique(y, return_counts=True)))
    print("Class counts:", counts)
    print("  0=bg  1=droplet  2=NPC  3=nucleus")


# Visually confirm nuclei are in focus and NPC class looks correct before training.
show_random_patch(cfg)

In [ ]:
# ============================================================
# 12. TensorFlow dataset — timepoint-stratified split + augmentation
# ============================================================

def augment_pair(image, label_oh):
    """Identical random flip + rotation applied to image and label."""
    n_img_ch = image.shape[-1]
    combined = tf.concat([image, label_oh], axis=-1)
    combined = tf.image.random_flip_left_right(combined)
    combined = tf.image.random_flip_up_down(combined)
    k        = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    combined = tf.image.rot90(combined, k=k)
    return combined[..., :n_img_ch], combined[..., n_img_ch:]


def load_npy_pair_py(img_path, lab_path):
    img_path = img_path.item() if isinstance(img_path, np.ndarray) else img_path
    lab_path = lab_path.item() if isinstance(lab_path, np.ndarray) else lab_path
    if isinstance(img_path, bytes): img_path = img_path.decode('utf-8')
    if isinstance(lab_path, bytes): lab_path = lab_path.decode('utf-8')
    return np.load(img_path).astype('float32'), np.load(lab_path).astype('int32')


def tf_load_npy_pair(img_path, lab_path):
    img_arr, lab_arr = tf.numpy_function(
        load_npy_pair_py, [img_path, lab_path], [tf.float32, tf.int32])
    img_arr.set_shape((cfg.patch_size, cfg.patch_size, cfg.n_channels))
    lab_arr.set_shape((cfg.patch_size, cfg.patch_size))
    return img_arr, tf.one_hot(lab_arr, depth=cfg.num_classes, dtype=tf.float32)


def make_dataset(img_paths, lab_paths, batch_size=cfg.batch_size,
                 shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (list(map(str, img_paths)), list(map(str, lab_paths))))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(img_paths), seed=cfg.seed,
                        reshuffle_each_iteration=True)
    ds = ds.map(tf_load_npy_pair, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def build_train_val_datasets(cfg=cfg):
    img_paths, lab_paths = list_patch_files(cfg)
    if len(img_paths) != len(lab_paths):
        raise ValueError(f'Image/label count mismatch: {len(img_paths)} vs {len(lab_paths)}')
    if len(img_paths) == 0:
        raise ValueError('No patches found. Run patch generation first.')

    val_timepoints = (list(cfg.val_timepoints) if cfg.val_timepoints is not None
                      else [img_fov.shape[0] - 1])
    val_t_strs = [f'_t{t:03d}_' for t in val_timepoints]
    print(f'Validation timepoints : {val_timepoints}')
    print(f'Augmentation          : {cfg.use_augmentation}')

    train_img, train_lab, val_img, val_lab = [], [], [], []
    for ip, lp in zip(img_paths, lab_paths):
        if '_cneg_' in ip.name:
            train_img.append(ip); train_lab.append(lp); continue
        if any(s in ip.name for s in val_t_strs):
            val_img.append(ip); val_lab.append(lp)
        else:
            train_img.append(ip); train_lab.append(lp)

    if len(val_img) == 0:
        raise ValueError(f'No patches matched val_timepoints={val_timepoints}.')

    print(f'Train patches : {len(train_img)}')
    print(f'Val patches   : {len(val_img)}')

    train_ds = make_dataset(train_img, train_lab, cfg.batch_size,
                            shuffle=True, augment=cfg.use_augmentation)
    val_ds   = make_dataset(val_img,   val_lab,   cfg.batch_size,
                            shuffle=False, augment=False)
    return train_ds, val_ds, train_img, val_img


train_ds, val_ds, train_img_paths, val_img_paths = build_train_val_datasets(cfg)

In [ ]:
# ============================================================
# 13. U-Net model
# ============================================================

def conv_block(x, filters, dropout_rate=0.0):
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    if dropout_rate > 0.0:
        x = layers.SpatialDropout2D(dropout_rate)(x)
    return x

def encoder_block(x, filters, dropout_rate=0.0):
    c = conv_block(x, filters, dropout_rate=dropout_rate)
    p = layers.MaxPooling2D((2, 2))(c)
    return c, p

def decoder_block(x, skip, filters, dropout_rate=0.0):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters, dropout_rate=dropout_rate)
    return x

def build_unet(input_shape=None, num_classes=None):
    if input_shape is None:
        input_shape = (cfg.patch_size, cfg.patch_size, cfg.n_channels)
    if num_classes is None:
        num_classes = cfg.num_classes

    inputs = layers.Input(shape=input_shape)

    c1, p1 = encoder_block(inputs, 32,  dropout_rate=0.0)
    c2, p2 = encoder_block(p1,     64,  dropout_rate=0.0)
    c3, p3 = encoder_block(p2,     128, dropout_rate=0.0)
    c4, p4 = encoder_block(p3,     256, dropout_rate=0.0)

    bn = conv_block(p4, 512, dropout_rate=0.5)

    d1 = decoder_block(bn, c4, 256, dropout_rate=0.3)
    d2 = decoder_block(d1, c3, 128, dropout_rate=0.3)
    d3 = decoder_block(d2, c2, 64,  dropout_rate=0.0)
    d4 = decoder_block(d3, c1, 32,  dropout_rate=0.0)

    # 4-class output: 0=bg, 1=droplet, 2=NPC, 3=nucleus
    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(d4)
    return models.Model(inputs, outputs, name=cfg.model_name)


model = build_unet()
model.summary()

In [ ]:
# ============================================================
# 14. Loss, metrics, callbacks, training
# ============================================================

_CLASS_WEIGHTS = tf.constant(list(cfg.loss_class_weights), dtype=tf.float32)


def weighted_dice_loss(y_true, y_pred, smooth=1e-6):
    """Per-class Dice weighted by loss_class_weights."""
    y_true_f       = tf.reshape(y_true, [-1, cfg.num_classes])
    y_pred_f       = tf.reshape(y_pred, [-1, cfg.num_classes])
    intersection   = tf.reduce_sum(y_true_f * y_pred_f, axis=0)
    denom          = tf.reduce_sum(y_true_f + y_pred_f, axis=0)
    dice_per_class = (2.0 * intersection + smooth) / (denom + smooth)
    weighted_mean  = (tf.reduce_sum(_CLASS_WEIGHTS * dice_per_class) /
                      tf.reduce_sum(_CLASS_WEIGHTS))
    return 1.0 - weighted_mean


def weighted_ce_loss(y_true, y_pred):
    """Categorical cross-entropy with per-pixel class weighting."""
    pixel_weights = tf.reduce_sum(_CLASS_WEIGHTS * y_true, axis=-1)
    ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    return tf.reduce_mean(pixel_weights * ce)


def combined_weighted_loss(y_true, y_pred):
    """Primary training loss: weighted CE + weighted Dice."""
    return weighted_ce_loss(y_true, y_pred) + weighted_dice_loss(y_true, y_pred)


def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """Unweighted mean Dice — monitoring metric."""
    y_true_f     = tf.reshape(y_true, [-1, cfg.num_classes])
    y_pred_f     = tf.reshape(y_pred, [-1, cfg.num_classes])
    intersection = tf.reduce_sum(y_true_f * y_pred_f, axis=0)
    denom        = tf.reduce_sum(y_true_f + y_pred_f, axis=0)
    return tf.reduce_mean((2.0 * intersection + smooth) / (denom + smooth))


def npc_dice(y_true, y_pred, smooth=1e-6):
    """Dice for NPC class (class 2) — primary detection metric."""
    y_true_npc   = y_true[..., 2]
    y_pred_npc   = y_pred[..., 2]
    intersection = tf.reduce_sum(y_true_npc * y_pred_npc)
    denom        = tf.reduce_sum(y_true_npc) + tf.reduce_sum(y_pred_npc)
    return (2.0 * intersection + smooth) / (denom + smooth)


def nucleus_dice(y_true, y_pred, smooth=1e-6):
    """Dice for nucleus interior (class 3) — measurement ROI metric."""
    y_true_nuc   = y_true[..., 3]
    y_pred_nuc   = y_pred[..., 3]
    intersection = tf.reduce_sum(y_true_nuc * y_pred_nuc)
    denom        = tf.reduce_sum(y_true_nuc) + tf.reduce_sum(y_pred_nuc)
    return (2.0 * intersection + smooth) / (denom + smooth)


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate),
    loss=combined_weighted_loss,
    metrics=[dice_coefficient, npc_dice, nucleus_dice, 'categorical_accuracy'],
)

callbacks = [
    # Monitor NPC Dice as primary — detects nuclei at all timepoints
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(cfg.best_model_path),
        monitor="val_npc_dice",
        save_best_only=True, mode='max', verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_npc_dice",
        factor=0.5, patience=4, min_delta=1e-4, mode='max', verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_npc_dice",
        patience=10, restore_best_weights=True, mode='max', verbose=1,
    ),
]

history = model.fit(train_ds, validation_data=val_ds,
                    epochs=cfg.epochs, callbacks=callbacks)
model.save(cfg.final_model_path)

In [ ]:
# ============================================================
# 15. Training history visualization
# ============================================================

def plot_training_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history.history["loss"],     label="Train loss")
    axes[0].plot(history.history["val_loss"], label="Val loss")
    axes[0].set_title("Combined weighted loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["dice_coefficient"],     label="Train mean Dice")
    axes[1].plot(history.history["val_dice_coefficient"], label="Val mean Dice")
    axes[1].plot(history.history["npc_dice"],             label="Train NPC Dice",    linestyle="--")
    axes[1].plot(history.history["val_npc_dice"],         label="Val NPC Dice",      linestyle="--")
    axes[1].plot(history.history["nucleus_dice"],         label="Train nucleus Dice", linestyle=":")
    axes[1].plot(history.history["val_nucleus_dice"],     label="Val nucleus Dice",   linestyle=":")
    axes[1].set_title("Dice metrics (NPC = primary, nucleus = secondary)")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    axes[2].plot(history.history["categorical_accuracy"],     label="Train accuracy")
    axes[2].plot(history.history["val_categorical_accuracy"], label="Val accuracy")
    axes[2].set_title("Categorical accuracy")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()

    plt.tight_layout()
    plt.show()


plot_training_history(history)

In [ ]:
# ============================================================
# 16. Visualize validation predictions
# ============================================================

def show_prediction_batch(model, dataset, n=2):
    for x_batch, y_batch in dataset.take(1):
        preds       = model.predict(x_batch)
        pred_labels = np.argmax(preds, axis=-1)
        true_labels = np.argmax(y_batch.numpy(), axis=-1)
        n_show      = min(n, x_batch.shape[0])

        for i in range(n_show):
            fig, axes = plt.subplots(2, 4, figsize=(20, 10))

            axes[0, 0].imshow(x_batch[i, ..., 0], cmap="gray");  axes[0, 0].set_title("Input: NLS (nucleus)")
            axes[0, 1].imshow(x_batch[i, ..., 1], cmap="gray");  axes[0, 1].set_title("Input: NPC")
            axes[0, 2].imshow(true_labels[i],  cmap="viridis", vmin=0, vmax=3); axes[0, 2].set_title("True label")
            axes[0, 3].imshow(pred_labels[i],  cmap="viridis", vmin=0, vmax=3); axes[0, 3].set_title("Predicted label")

            axes[1, 0].imshow(preds[i, ..., 0], cmap="inferno", vmin=0, vmax=1); axes[1, 0].set_title("Background prob")
            axes[1, 1].imshow(preds[i, ..., 1], cmap="inferno", vmin=0, vmax=1); axes[1, 1].set_title("Droplet prob")
            axes[1, 2].imshow(preds[i, ..., 2], cmap="inferno", vmin=0, vmax=1); axes[1, 2].set_title("NPC prob")
            axes[1, 3].imshow(preds[i, ..., 3], cmap="inferno", vmin=0, vmax=1); axes[1, 3].set_title("Nucleus prob")

            for a in axes.ravel():
                a.axis("off")
            plt.tight_layout()
            plt.show()
        break


show_prediction_batch(model, val_ds, n=2)